In [104]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [105]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

def find_project_root(start: Path, marker: str = "src") -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / marker).exists():
            return path
    raise RuntimeError(f"Could not find project root containing '{marker}'")

PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = C:\Users\rawls.varghese\quant-lab


**Modelling Long-Time Horizon**

In [106]:
df_v2 = pd.read_csv("../../../data/processed/project_04_features_v2.csv")
df_v2["Date"] = pd.to_datetime(df_v2["Date"])
df_v2 = df_v2.sort_values(["Date", "Ticker"]).copy()

df_v2.head()

,Date,Ticker,ret_1,ret_5,ret_10,ret_20,ret_60,ret_120,vol_5,vol_20,ma_10_ratio,ma_20_ratio,ma_50_ratio,dollar_vol,vol_z_20,fwd_ret_5
0,1985-06-24,AAPL,0.069551,0.162329,0.069551,-0.048546,-0.212334,-0.382087,0.022720,0.040407,0.102600,0.061530,-0.093769,1.580152e+07,-0.950853,0.051022
37143,1985-06-24,CVX,0.000000,0.006962,0.028120,0.033018,0.075958,0.250063,0.010172,0.008068,0.011799,0.021790,0.029220,1.774198e+06,-1.210953,0.015858
59324,1985-06-24,HD,0.000000,0.000000,-0.114053,-0.162168,-0.183514,-0.162168,0.038925,0.027756,-0.036986,-0.086747,-0.161552,6.654329e+05,-0.374105,-0.097324
69576,1985-06-24,JNJ,0.009080,0.018519,0.009080,0.009080,0.170248,0.349638,0.005049,0.010789,0.019438,0.011472,0.044918,8.655933e+06,-0.755954,-0.027430
79829,1985-06-24,JPM,0.003164,0.009508,-0.009416,-0.018575,0.079978,0.204943,0.012172,0.009426,0.000291,-0.006279,-0.000026,2.891721e+06,-0.272785,0.018927


**Features and Target**

In [107]:
feature_cols_v2 = ["ret_1", "ret_5", "ret_10", "ret_20", "ret_60",
                    "ret_120", "vol_5", "vol_20", "ma_10_ratio",
                      "ma_20_ratio", "ma_50_ratio", "dollar_vol",
                        "vol_z_20"
                        ]

target_v2 = "fwd_ret_5"

X = df_v2[feature_cols_v2].copy()
y = df_v2[target_v2].copy()

**Time Series Split**

In [108]:
split_date_v2 = "2016-01-01"

train_mask_v2 = df_v2["Date"] < split_date_v2
test_mask_v2 = df_v2["Date"] >= split_date_v2

X_v2_train = X.loc[train_mask_v2]
y_v2_train = y.loc[train_mask_v2]

X_v2_test = X.loc[test_mask_v2]
y_v2_test = y.loc[test_mask_v2]

df_v2_train = df_v2.loc[train_mask_v2].copy()
df_v2_test = df_v2.loc[test_mask_v2].copy()

print("Train_v2 Shape", X_v2_train.shape)
print("Test_v2 Shape", X_v2_test.shape)
print("Train Date Range:", df_v2_train["Date"].min(), df_v2_train["Date"].max())


Train_v2 Shape (122941, 13)
Test_v2 Shape (51160, 13)
Train Date Range: 1985-06-24 00:00:00 2015-12-31 00:00:00


In [109]:
linreg_v2 = LinearRegression()

linreg_v2.fit(X_v2_train, y_v2_train)
pred_train_v2 = linreg_v2.predict(X_v2_train)
pred_test_v2 = linreg_v2.predict(X_v2_test)

df_v2_train["pred"] = pred_train_v2
df_v2_test["pred"] = pred_test_v2

df_v2 = pd.concat([df_v2_train, df_v2_test]).sort_values(["Date", "Ticker"])
df_v2.head()

,Date,Ticker,ret_1,ret_5,ret_10,ret_20,ret_60,ret_120,vol_5,vol_20,ma_10_ratio,ma_20_ratio,ma_50_ratio,dollar_vol,vol_z_20,fwd_ret_5,pred
0,1985-06-24,AAPL,0.069551,0.162329,0.069551,-0.048546,-0.212334,-0.382087,0.022720,0.040407,0.102600,0.061530,-0.093769,1.580152e+07,-0.950853,0.051022,-0.006690
37143,1985-06-24,CVX,0.000000,0.006962,0.028120,0.033018,0.075958,0.250063,0.010172,0.008068,0.011799,0.021790,0.029220,1.774198e+06,-1.210953,0.015858,0.003879
59324,1985-06-24,HD,0.000000,0.000000,-0.114053,-0.162168,-0.183514,-0.162168,0.038925,0.027756,-0.036986,-0.086747,-0.161552,6.654329e+05,-0.374105,-0.097324,0.005138
69576,1985-06-24,JNJ,0.009080,0.018519,0.009080,0.009080,0.170248,0.349638,0.005049,0.010789,0.019438,0.011472,0.044918,8.655933e+06,-0.755954,-0.027430,0.003419
79829,1985-06-24,JPM,0.003164,0.009508,-0.009416,-0.018575,0.079978,0.204943,0.012172,0.009426,0.000291,-0.006279,-0.000026,2.891721e+06,-0.272785,0.018927,0.004271


**Error Calculation on Long Time Horizon**

In [110]:
rmse_v2_train = mean_squared_error(y_v2_train, pred_train_v2)
rmse_v2_test = mean_squared_error(y_v2_test, pred_test_v2)

mae_v2_train = mean_absolute_error(y_v2_train, pred_train_v2)
mae_v2_test = mean_absolute_error(y_v2_test, pred_test_v2)

print("Train Mean Squared Error:", rmse_v2_train)
print("Test Mean Squared Error:", rmse_v2_test)

print("Train Mean Squared Error:", mae_v2_train)
print("Test Mean Absolute Error:", mae_v2_test)


Train Mean Squared Error: 0.0024697649981162777
Test Mean Squared Error: 0.0018866489877833784
Train Mean Squared Error: 0.0330605261467198
Test Mean Absolute Error: 0.029759080794917805


**Correlation**

In [111]:
v2_ic = df_v2.groupby("Date").apply(lambda x: x["pred"].corr(x["fwd_ret_5"]) if x["pred"].std() > 0 else np.nan)
print("IC Mean:", v2_ic.mean())

IC Mean: 0.025545426722455678


**Quantiles**


In [112]:
df_v2["pred_q"] = df_v2.groupby("Date")["pred"].transform(lambda x: pd.qcut(x, 5, labels=False, duplicates="drop"))
df_v2.groupby("pred_q")["fwd_ret_5"].mean()

pred_q
0.0    0.003623
1.0    0.003442
2.0    0.003087
3.0    0.004165
4.0    0.006365
Name: fwd_ret_5, dtype: float64

The extended model incorporating long-horizon features shows a significant improvement in predictive power, with mean IC increasing from ~0.02 to ~0.04 and a clearer separation across quintiles, indicating that combining short-term mean reversion with weak long-term momentum enhances cross-sectional ranking performance.

**Long Short Portfolio**

**Quantiles on test data**

In [113]:
df_v2_test["pred_q"] = df_v2_test.groupby("Date")["pred"].transform(lambda x: pd.qcut(x, 5, labels=False, duplicates="drop"))
df_v2_test.head()

,Date,Ticker,ret_1,ret_5,ret_10,ret_20,ret_60,ret_120,vol_5,vol_20,ma_10_ratio,ma_20_ratio,ma_50_ratio,dollar_vol,vol_z_20,fwd_ret_5,pred,pred_q
7696,2016-01-04,AAPL,0.000859,-0.024838,-0.033062,-0.085454,-0.044787,-0.153737,0.014727,0.015968,-0.015962,-0.048424,-0.083016,6.620124e+09,1.035662,-0.064857,0.001551,0
14817,2016-01-04,AMZN,-0.057554,-0.038926,-0.050190,-0.043917,0.175388,0.368194,0.033817,0.020630,-0.047764,-0.044952,-0.027994,5.258779e+09,2.576012,-0.030220,0.006377,3
24714,2016-01-04,BAC,-0.023640,-0.048560,-0.050174,-0.050174,0.046269,-0.034978,0.011785,0.019195,-0.034258,-0.041841,-0.045548,1.886168e+09,0.901005,-0.068093,0.006253,3
34585,2016-01-04,COST,-0.012211,-0.013662,-0.007128,-0.025408,0.081010,0.114303,0.008179,0.016229,-0.008692,-0.015307,-0.003603,3.687279e+08,-0.079286,-0.030191,0.004944,1
44836,2016-01-04,CVX,-0.012374,-0.034775,-0.018674,0.000000,0.020561,-0.046889,0.011251,0.021751,-0.019004,-0.011994,-0.018098,1.011358e+09,-0.177479,-0.090928,0.005406,2


**Long and Short leg Returns per day**

In [114]:
v2_daily_portfolio = df_v2_test.groupby("Date").apply(
    lambda x: pd.Series({
        "long_ret": x.loc[x["pred_q"] ==  x["pred_q"].max(), "fwd_ret_5"].mean(),
        "short_ret": x.loc[x["pred_q"] == x["pred_q"].min(), "fwd_ret_5"].mean()
})
)
v2_daily_portfolio.head()

,long_ret,short_ret
Date,,
2016-01-04,-0.053290,-0.028576
2016-01-05,-0.049970,-0.022447
2016-01-06,-0.069095,-0.038790
2016-01-07,-0.016933,0.007567
2016-01-08,-0.031673,-0.010705


**Long Short Spread**

In [115]:
v2_daily_portfolio["ls_ret_5d"] = v2_daily_portfolio["long_ret"] - v2_daily_portfolio["short_ret"]
v2_daily_portfolio.head()

,long_ret,short_ret,ls_ret_5d
Date,,,
2016-01-04,-0.053290,-0.028576,-0.024714
2016-01-05,-0.049970,-0.022447,-0.027524
2016-01-06,-0.069095,-0.038790,-0.030305
2016-01-07,-0.016933,0.007567,-0.024500
2016-01-08,-0.031673,-0.010705,-0.020968


**Daily Approximation**

In [116]:
v2_daily_portfolio["ls_ret_daily"] = v2_daily_portfolio["ls_ret_5d"] / 5
v2_daily_portfolio.head()

,long_ret,short_ret,ls_ret_5d,ls_ret_daily
Date,,,,
2016-01-04,-0.053290,-0.028576,-0.024714,-0.004943
2016-01-05,-0.049970,-0.022447,-0.027524,-0.005505
2016-01-06,-0.069095,-0.038790,-0.030305,-0.006061
2016-01-07,-0.016933,0.007567,-0.024500,-0.004900
2016-01-08,-0.031673,-0.010705,-0.020968,-0.004194


The initial long-short portfolio produced consistently negative returns, indicating that the model’s predicted ranking was effectively inverted, with top-ranked stocks underperforming and bottom-ranked stocks outperforming.

In [117]:
from src.signals.ranking import standardize_signal
from src.portfolio.construction import build_daily_weights_from_panel
from src.backtest.engine import backtest_cross_sectional_strategy

df_v2 = standardize_signal(df_v2, "pred") #standardization turns your model from noisy regression output into stable cross sectional ranking signal. Considering ranking not magnitude

weights_v2 = build_daily_weights_from_panel(
    df_v2,
    signal_col="signal_z",
    date_col="Date",
    asset_col="Ticker"
)

port_v2 = backtest_cross_sectional_strategy(
    weights_v2,
    df_v2,
    return_col="fwd_ret_5"
)

In [118]:
from src.utils.metrics import sharpe_ratio
from src.utils.metrics import max_drawdown

print("Sharpe:", sharpe_ratio(port_v2["portfolio_return"]))
print("MDD:", max_drawdown(port_v2["equity_curve"]))

Sharpe: 1.191450810337205
MDD: -0.9056535802444304


The v2 signal demonstrates strong cross-sectional alpha with a Sharpe ratio of ~1.19, but suffers from extremely high drawdown (~-90%), indicating that while the ranking signal is effective, the current portfolio construction lacks risk control and leads to unstable performance.